# 06.6 — Transformer Decoder

**The decoder stack** adds masked self-attention on top of the encoder architecture. The causal mask is what makes autoregressive generation possible.

**Decoder layer = 3 sublayers:**
1. Masked self-attention (can't see future)
2. Cross-attention on encoder output (for seq2seq; absent in decoder-only GPT)
3. Feed-forward network

**Decoder-only vs encoder-decoder:** GPT uses decoder-only (just masked self-attention). T5, BART use full encoder-decoder.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Reuse components from previous notebooks
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = F.softmax(scores, dim=-1)
    return torch.nan_to_num(torch.matmul(attn, V)), attn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_k = d_model // n_heads
        self.n_heads = n_heads; self.d_model = d_model
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
    def split_heads(self, x):
        b,s,_ = x.shape
        return x.view(b,s,self.n_heads,self.d_k).transpose(1,2)
    def forward(self, Q, K, V, mask=None):
        b = Q.size(0)
        Q,K,V = self.split_heads(self.W_Q(Q)), self.split_heads(self.W_K(K)), self.split_heads(self.W_V(V))
        out,attn = scaled_dot_product_attention(Q,K,V,mask)
        return self.W_O(out.transpose(1,2).contiguous().view(b,-1,self.d_model)), attn

class DecoderLayer(nn.Module):
    """
    Transformer decoder layer with:
    1. Masked self-attention (causal)
    2. Cross-attention on encoder output (optional, for seq2seq)
    3. Feed-forward
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, use_cross_attention=True):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.use_cross_attn = use_cross_attention
        if use_cross_attention:
            self.cross_attn = MultiHeadAttention(d_model, n_heads)
            self.norm_cross = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, encoder_output=None, src_mask=None, tgt_mask=None):
        # 1. Masked self-attention
        attn_out, _ = self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), tgt_mask)
        x = x + self.dropout(attn_out)
        
        # 2. Cross-attention (encoder-decoder attention)
        if self.use_cross_attn and encoder_output is not None:
            norm_x = self.norm_cross(x)
            cross_out, _ = self.cross_attn(norm_x, encoder_output, encoder_output, src_mask)
            x = x + self.dropout(cross_out)
        
        # 3. Feed-forward
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x

class CausalTransformer(nn.Module):
    """
    Decoder-only transformer (GPT-style): just masked self-attention.
    Input tokens -> predict next tokens.
    """
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4, 
                 d_ff=512, max_len=256, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout, use_cross_attention=False)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.max_len = max_len
    
    def _make_causal_mask(self, seq_len, device):
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
        return mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq, seq]
    
    def forward(self, x):
        b, s = x.shape
        positions = torch.arange(s, device=x.device)
        h = self.embedding(x) + self.pos_emb(positions)
        
        mask = self._make_causal_mask(s, x.device)
        for layer in self.layers:
            h = layer(h, tgt_mask=mask)
        
        return self.lm_head(self.norm(h))
    
    @torch.no_grad()
    def generate(self, prompt: list[int], max_new_tokens=50, temperature=1.0, top_k=10):
        self.eval()
        tokens = torch.tensor([prompt])
        
        for _ in range(max_new_tokens):
            if tokens.size(1) >= self.max_len:
                break
            logits = self(tokens)[:, -1, :]  # last token's logits
            logits = logits / temperature
            
            # Top-k sampling
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            tokens = torch.cat([tokens, next_token], dim=1)
        
        return tokens[0].tolist()

# Build a small GPT-like model
gpt = CausalTransformer(vocab_size=100, d_model=64, n_heads=4, n_layers=2)
n_params = sum(p.numel() for p in gpt.parameters())
print(f"CausalTransformer params: {n_params:,}")

# Forward pass
x = torch.randint(0, 100, (2, 20))
logits = gpt(x)
print(f"Input: {x.shape} -> Logits: {logits.shape}")
print(f"Each position predicts distribution over {logits.shape[-1]} tokens")

In [ ]:
# The causal mask: why it's critical

print("Causal mask:")
mask = torch.tril(torch.ones(5, 5))
print(mask.numpy().astype(int))
print()
print("Position 0: can only attend to position 0")
print("Position 2: can attend to positions 0, 1, 2")
print("Position 4: can attend to all previous positions")
print()
print("Why? At training time, we have all tokens.")
print("Without the mask, position 0 could 'cheat' by looking at future tokens.")
print("At inference: tokens are generated one at a time (causal = correct)")
print()
print("This means: a Transformer trained with causal masking can be")
print("parallelized during TRAINING (compute all positions at once),")
print("but is sequential during INFERENCE (generate one token at a time).")

## Summary

**Decoder-only (GPT-style):** Masked self-attention only. Simpler, scales better. Used for language modeling and generation.

**Encoder-decoder (T5-style):** Full encoder + decoder with cross-attention. Better for seq2seq tasks (translation, summarization) where input and output sequences are distinct.

**KV-Cache:** During inference, the keys and values from previous steps don't change. Cache them to avoid recomputation → makes generation much faster.

---
**Next:** 06.7 — Mini-GPT trained on real text